In [ ]:
import os
import sys
import xarray as xr
import netCDF4 as nc
import pandas
import numpy as np
import glob
import pandas as pd

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns

# settings
%config InlineBackend.figure_format = 'retina'

# top level directory
dpath0='/Users/dervlamk/OneDrive/research/eastern_pacific_cores'
# save figs here
opath=f'{dpath0}/../nam/PaleoPaleo2025_manuscript'

**Bacon age models**

In [ ]:
# DSDP-480 depths
dsdp480_depths=pd.read_excel(f'{dpath1}/sample_depths_480.xlsx').depth.values # cm

# DSDP-480 bacon inputs
dsdp480_input=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP480/DSDP480.csv')
xerr480 = dsdp480_input.error.values  # x-axis error

# DSDP-480 bacon ensem
dsdp480_mcmc=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP480/DSDP480_mcmc_new.csv', header=None)
dsdp480_mcmc_sorted=np.sort(dsdp480_mcmc, axis=0) # sort, each row is an age model
ens_num=np.linspace(1,len(dsdp480_mcmc_sorted),len(dsdp480_mcmc_sorted))
dims = ['ensemble_number','depth']
coords = {'ensemble_number': ens_num,
          'depth': dsdp480_depths}
dsdp480_agedepth=xr.DataArray(dsdp480_mcmc_sorted, dims=dims, coords=coords)
median480=dsdp480_agedepth.median(axis=0)

In [ ]:
new_median = median480.to_series().reset_index().drop_duplicates(subset='depth').set_index('depth').to_xarray()
new_median.interp(depth=4307) # pollen tie point
new_median.interp(depth=4596) # dDwax tie point

# age, depth
d480_479_tiepoint = [[108572.80059, 4307], # DSDP-480-479 pollen tie point
                     [118676.66515, 4596]] # DSDP-480-479 dDwax tie point
#[110134.71, 4351],

In [ ]:
# DSDP-479 depths
dsdp479_depths=pd.read_excel(f'{dpath1}/sample_depths_479.xlsx').depth.values # cm

# DSDP-479 bacon inputs
dsdp479_input=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP479/DSDP479.csv')
xerr479 = dsdp479_input.error.values  # x-axis error

# DSDP-479 bacon ensem
dsdp479_mcmc=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP479/DSDP479_mcmc.csv', header=None)
dsdp479_mcmc_sorted=np.sort(dsdp479_mcmc, axis=0) # sort, each row is an age model
ens_num=np.linspace(1,len(dsdp479_mcmc_sorted),len(dsdp479_mcmc_sorted))
dims = ['ensemble_number','depth']
coords = {'ensemble_number': ens_num,
          'depth': dsdp479_depths}
dsdp479_agedepth=xr.DataArray(dsdp479_mcmc_sorted, dims=dims, coords=coords)
median479=dsdp479_agedepth.median(axis=0)

In [ ]:
median479.interp(depth=3476) # pollen tie point
median479.interp(depth=4351) # dDwax tie point

# age, depth
d479_480_tiepoint = [[106559.5, 3476], # DSDP-480-479 pollen tie point
                     [124952.8383, 4351]] # DSDP-480-479 dDwax tie point

In [ ]:
iters = 4000
# 95% two-tailed
iupperupper = int(iters * 0.025)
ilowerlower = int(iters * 0.975)

# 90% two-tailed
iupper = int(iters * 0.05)
ilower = int(iters * 0.95)

# 68% two-tailed
iouter = int(iters * 0.16)
iinner = int(iters * 0.84)

something is wrong with the code below... doesn't make sense that the age model isn't lining up with the 14C points

In [ ]:
line_kw={'ls':'-', 'lw':2}
err_line_kw={'ls':':', 'lw':0.75, 'color':'k'}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'clip_on': False, 'zorder':100}
tkw = {'axis':'both', 'direction':'in', 'labelsize': 10}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':11, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
axis_text_kw={'weight':'normal', 'size':11, 'color':'k'}
legend_kw = {'loc':3, 'fontsize':8, 'labelcolor':'k', 'frameon':False}

fig = plt.figure(figsize=(9,6))

# DSDP-480
ax = plt.subplot(121)
# 95% confidence interval
ax.plot(dsdp480_agedepth[iupperupper, :], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(dsdp480_agedepth[ilowerlower, :], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
# 2-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 dsdp480_agedepth[ilowerlower, :],
                 dsdp480_agedepth[iupperupper, :],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence')
# 1-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 dsdp480_agedepth[iouter, :],
                 dsdp480_agedepth[iinner, :],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence')               
# median line
ax.plot(median480, dsdp480_agedepth.depth, '-', c='k', lw=1, label='median')
# ms tie-points
plt.errorbar(dsdp480_input.iloc[1:6].age.values, dsdp480_input.iloc[1:6].depth.values, 
             xerr=xerr480[1:6]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[1:6].age.values, dsdp480_input.iloc[1:6].depth.values,
            marker='s', color='lightblue', **scat_kw, label='M.S. tie')
# planktic D14C
plt.errorbar(dsdp480_input.iloc[6:12].age.values-dsdp480_input.iloc[6:12].dR.values, dsdp480_input.iloc[6:12].depth.values, 
             xerr=xerr480[6:12]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[6:12].age.values, dsdp480_input.iloc[6:12].depth.values,
            marker='^', color='white', **scat_kw, label='$\Delta^{14}$C$_{planktic}$')
# benthic d18O tie points
plt.errorbar(dsdp480_input.iloc[12:14].age.values-dsdp480_input.iloc[12:14].dR.values, dsdp480_input.iloc[12:14].depth.values, 
             xerr=xerr480[12:14]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[12:14].age.values, dsdp480_input.iloc[12:14].depth.values,
            marker='^', color='lightblue', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie')
# d479 tie points
plt.errorbar(dsdp480_input.iloc[14:16].age.values-dsdp480_input.iloc[14:16].dR.values, dsdp480_input.iloc[14:16].depth.values, 
             xerr=xerr480[14:16]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[14:16].age.values, dsdp480_input.iloc[14:16].depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta$D$_{C30}$ tie')
# benthic d18O tie points
plt.errorbar(dsdp480_input.iloc[16].age-dsdp480_input.iloc[16].dR, dsdp480_input.iloc[16].depth, 
             xerr=xerr[16]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[16].age, dsdp480_input.iloc[16].depth,
            marker='^', color='lightblue', **scat_kw, label='_Hidden')
# DSDP-479 tie-points
plt.scatter(d480_479_tiepoint[1][0], d480_479_tiepoint[1][1],
            marker='x', color='r', **scat_kw, label='DSDP-479/480 $\delta$D$_{C30}$ tie')
plt.scatter(d480_479_tiepoint[0][0], d480_479_tiepoint[0][1],
                marker='x', color='coral', **scat_kw, label='DSDP-479/480 pollen tie')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.set(xlim=[1,137000], ylim=[5000,0])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.set_ylabel("Core Depth (cm)")
ax.set_xticks([25000,50000,75000,100000,125000])
ax.set_xticklabels([25,50,75,100,125])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False, **tkw)
ax.legend(**legend_kw)


# DSDP-479
ax = plt.subplot(122)
# 2-sigma bounds and shading
ax.plot(dsdp479_agedepth[iupperupper, :], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(dsdp479_agedepth[ilowerlower, :], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.fill_betweenx(dsdp479_agedepth.depth,
                 dsdp479_agedepth[ilowerlower, :],
                 dsdp479_agedepth[iupperupper, :],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence interval')
# 1-sigma shading
ax.fill_betweenx(dsdp479_agedepth.depth,
                 dsdp479_agedepth[iouter, :],
                 dsdp479_agedepth[iinner, :],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence interval')
# median line
ax.plot(median479, dsdp479_agedepth.depth, '-', c='k', lw=1, label='median')
# d479-LR04 tie points
plt.errorbar(dsdp479_input.age.values-dsdp479_input.dR.values, dsdp479_input.depth.values, 
             xerr=xerr479*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp479_input.age.values, dsdp479_input.depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta$D$_{C30}$ tie')
# DSDP-480 tie-points
plt.scatter(d479_480_tiepoint[0][0], d479_480_tiepoint[0][1],
                marker='x', color='coral', **scat_kw, label='DSDP-480 pollen tie') #
plt.scatter(d479_480_tiepoint[1][0], d479_480_tiepoint[1][1],
            marker='x', color='r', **scat_kw, label='DSDP-480 $\delta$D$_{C30}$ tie') #
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.set(xlim=[100000,160000], ylim=[5750,3450])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.yaxis.set_label_position('right')
ax.set_ylabel(" ", rotation=270, labelpad=15)
ax.set_xticks([100000,120000,140000,160000])
ax.set_xticklabels([100,120,140,160])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False,
               right=True, labelright=True, **tkw)

plt.savefig(f'{opath}/dsdp_480-479.agemodel.pdf', bbox_inches='tight')

**Age model relevant proxy data on ages**

---
Shackleton & Hall, 1982 $\delta^{18}O$<br>
OXYGEN ISOTOPE STUDY OF CONTINUOUS SCRAPE SAMPLES FROM SITE 480<br>
DOI: 10.2973/dsdp.proc.64.165.1982

Keigwin & Jones, 1990 $\delta^{18}O$<br>
DOI:

Bryne et al., 1990 pollen<br>
DOI:

Lisiecki & Raymo, 2004 $\delta^{18}O$<br>
DOI:

In [ ]:
dpath1=f'{dpath0}/DSDP-480-479/age_model'

# Shackleton & Hall d18O
filen=f'{dpath1}/ShackletonHall82_d18O.xlsx'
sh_d18o={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').d18o.values
sh_d18o['dat'] = xr.DataArray(data=dat,
                              coords={'depth': depth},
                              dims='depth',
                              name='d18o')
sh_d18o['dat'].attrs['source'] = 'Shackleton_Hall_1982'

# Keigwin & Jones d18O
filen=f'{dpath1}/KeigwinJones90_d18O.xlsx'
kj_d18o={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').d18o.values
kj_d18o['dat']=xr.DataArray(data=dat,
                            coords={'depth': depth},
                            dims='depth',
                            name='d18o')
kj_d18o['dat'].attrs['source'] = 'Keigwin_Jones_1990'

# Byrne et al. pollen
filen=f'{dpath1}/Byrne90_pollen.xlsx'
by_aj={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').ArtemisiaJuniper.values
by_aj['dat']= xr.DataArray(data=dat,
                           coords={'depth': depth},
                           dims='depth',
                           name='ArtemisiaJuniper')
by_aj['dat'].attrs['source'] = 'Byrne_1990'

# LR04 benthic stack
filen=f'{dpath0}/../global_paleo_data/LR04stack_d18O.csv'
lr04={}
lr04['age']=pd.read_csv(filen).time.values
lr04['dat']=pd.read_csv(filen).d18O.values

hol_d18o_max = 2.1
hol_d18o_min = 2.5

In [ ]:
tie_points = {}

# in matlab code, age is listed as: 126.833 ka
tie_points['d18o_this_study']= xr.Dataset({"d18o": (("age"), [2.62, 2.43]),},
                               coords={"age": [130, 130], 
                                       "depth": [48.39, 48.39]})

# I don't think the ages on these reflect
tie_points['d18o']= xr.Dataset({"lr04_d18o": (("age"), [4.41, 4.47, 3.67]),
                                "sh_d18o": (("depth"), [3.6, 3.97, 2.48])},
                               coords={"age": [38, 69, 130],
                                       "depth": [23.95, 34.45, 47.90]})
d480_ages = new_median.interp(depth=[2395, 3445, 4790]).to_array(name='age').squeeze()/1000 #[39.395,  71.990  , 123.611]
tie_points['dD']= xr.Dataset({"lr04_d18o": (("age"), [4.12,3.1,3.14,3.67,4.86]),
                                "dsdp480_dD": (("depth"), [-157.26, -142.15, -141.77, -136.01, -149.6, -159.45])},
                               coords={"age": [109,123,125,130,135],
                                       "depth": [37.11, 41.81, 43.15, 45.96, 46.15, 48.11]})

Honestly, I'm not sure how I got the pollen all on one age model before. I need a combined age-depth model for DSDP 480 & 479, but it's unclear to me how to combine the depths for the two cores appropriately. Based on an old matlab code, I suggest that the 479 depths are 8.3 m (830 cm) higher than 480 such that adding 8.3 m to the 479 depths will bring them in line with 480. 

Also, I don't know what the deal is with the SH82 tie-points. Because all the timeseries are plotted vs. age, they should overlap precisely if there is a perfect match. But, given the 2000 year uncertainty range that we used for the Bacon model, there is some offset. But, previously I had the SH d18O measurement from 47.9 plotting right nearby the d18O data produced as part of this study. I don't know how I did that.

especially how I had the MIS5 one plotting right with our new data before. 

In [ ]:
#d480_ages = 
median479.interp(depth=[2395, 3445, 4790])/1000 #.to_array(name='age').squeeze()/1000 #[39.395,  71.990  , 123.611]


In [ ]:
# interp d18O and pollen depths to age model
sh_d18o['age']=new_median.interp(depth=sh_d18o['dat'].depth).to_array(name='age').squeeze()
kj_d18o['age']=new_median.interp(depth=kj_d18o['dat'].depth).to_array(name='age').squeeze()
by_aj['age']=new_median.interp(depth=by_aj['dat'].depth).to_array(name='age').squeeze()

In [ ]:
line_kw={'ls':'-', 'lw':1.5} #, 'marker':'s', 'markersize':4, 'mec':'k', 'mew':0.25, 'zorder':100} 
patch_kw = {'ec':'k', 'lw':1, 'linestyle':':', 'fc':'grey', 'alpha':0.25}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'zorder': 100, 'clip_on': False}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
arrow_kw={'arrowstyle':'->', 'color':'k', 'linewidth':2, 'clip_on':False}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':8, 'weight':'bold', 'color':'k', 'ha':'left', 'va':'center'} #'backgroundcolor':'white', 
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'k'}
legend_kw = {'loc':'upper right', 'bbox_to_anchor':(1.35, 1.1), 'fontsize':8, 'labelcolor':'k', 'frameon':False}
#settings
xmin=0
xmax=160



fig = plt.figure(figsize=(9,5))

# DSDP-480
ax1 = plt.subplot(111)
#ax.text(1, -23, 'DSDP-480/479', **title_text_kw)
pp=plt.Rectangle((0, hol_d18o_min), 11.7, hol_d18o_max-hol_d18o_min, zorder=100, label='_Hidden', clip_on=True, **patch_kw) 
ax1.add_patch(pp)
ax1.annotate('Expected\nHolocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$',
             xy=(11.7,2.3), xytext=(25,2.3),
             arrowprops=arrow_kw, **label_text_kw)
#ax1.annotate('Expected\nHolocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$',
#             xy=(0,hol_d18o_min), xytext=(0,hol_d18o_max-.05),
#             arrowprops=arrow_kw, **label_text_kw, zorder=100)
ax1.plot(sh_d18o['age']/1000, sh_d18o['dat'], c='lightblue', **line_kw, label='SH82 $\delta^{18}$O$_{benthic}$')
ax1.plot(kj_d18o['age']/1000, kj_d18o['dat'], c='lightsteelblue', **line_kw, label='KJ90 $\delta^{18}$O$_{benthic}$')
ax1.plot(lr04['age'], lr04['dat'], c='k', lw=2, label='LR04 $\delta^{18}$O$_{benthic}$') # benthic stack
ax1.scatter(tie_points['d18o_this_study'].d18o.age, tie_points['d18o_this_study'].d18o,
            marker='o', color='peru', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie-point, this study')
ax1.errorbar(tie_points['d18o_this_study'].d18o.age, tie_points['d18o_this_study'].d18o, 
             xerr=0, yerr=0.05, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
ax1.scatter(d480_ages, tie_points['d18o'].sh_d18o,
            marker='x', color='red', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie-point, SH82')
ax1.scatter(tie_points['d18o'].lr04_d18o.age, tie_points['d18o'].lr04_d18o,
            marker='x', color='red', **scat_kw, label='_Hidden')
ax1.set(xlim=[xmin,xmax], ylim=[5.2,2.1])
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.set_ylabel(u'$\delta^{18}O_{benthic}$ [‰]', labelpad=5, **laxis_text_kw)
ax1.set_xlabel('AGE (ka)')

ax2=ax1.twinx() 
l6, =ax2.plot(by_aj['age']/1000, by_aj['dat'], c='grey', ls='-.', lw=1.25, label='Byrne90') # art+jun pollen
ax2.set(xlim=[xmin,xmax], ylim=[33.5,-2], yticks=[0,10,20,30])
ax2.minorticks_on()
ax2.set_ylabel(u'%Artemisia+Juniper', labelpad=15, **raxis_text_kw)
ax2.patch.set_visible(False)
#ax2.spines['bottom'].set_color('none')

ax1.legend(**legend_kw)

## gridspec test

In [ ]:
line_kw={'ls':'-', 'lw':2}
err_line_kw={'ls':':', 'lw':0.75, 'color':'k'}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'clip_on': False, 'zorder':100}
tkw = {'axis':'both', 'direction':'in', 'labelsize': 10}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'top', 'ha':'right'}
label_text_kw={'size':11, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
axis_text_kw={'weight':'normal', 'size':11, 'color':'k'}
legend_kw = {'loc':3, 'fontsize':7, 'labelcolor':'k', 'frameon':False}

fig = plt.figure(figsize=(8,9), constrained_layout=True)

# DSDP-480
ax = plt.subplot2grid((3, 2), (0, 0), rowspan=2, colspan=1)
ax.text(-5000, -300, 'A', **title_text_kw)
# 95% confidence interval
ax.plot(dsdp480_agedepth[iupperupper, :], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(dsdp480_agedepth[ilowerlower, :], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
# 2-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 dsdp480_agedepth[ilowerlower, :],
                 dsdp480_agedepth[iupperupper, :],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence')
# 1-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 dsdp480_agedepth[iouter, :],
                 dsdp480_agedepth[iinner, :],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence')               
# median line
ax.plot(median480, dsdp480_agedepth.depth, '-', c='k', lw=1, label='median')
# ms tie-points
plt.errorbar(dsdp480_input.iloc[1:6].age.values, dsdp480_input.iloc[1:6].depth.values, 
             xerr=xerr480[1:6]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[1:6].age.values, dsdp480_input.iloc[1:6].depth.values,
            marker='s', color='lightblue', **scat_kw, label='M.S. tie')
# planktic D14C
plt.errorbar(dsdp480_input.iloc[6:12].age.values-dsdp480_input.iloc[6:12].dR.values, dsdp480_input.iloc[6:12].depth.values, 
             xerr=xerr480[6:12]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[6:12].age.values, dsdp480_input.iloc[6:12].depth.values,
            marker='^', color='white', **scat_kw, label='$\Delta^{14}$C$_{planktic}$')
# benthic d18O tie points
plt.errorbar(dsdp480_input.iloc[12:14].age.values-dsdp480_input.iloc[12:14].dR.values, dsdp480_input.iloc[12:14].depth.values, 
             xerr=xerr480[12:14]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[12:14].age.values, dsdp480_input.iloc[12:14].depth.values,
            marker='^', color='lightblue', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie')
# d479 tie points
plt.errorbar(dsdp480_input.iloc[14:16].age.values-dsdp480_input.iloc[14:16].dR.values, dsdp480_input.iloc[14:16].depth.values, 
             xerr=xerr480[14:16]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[14:16].age.values, dsdp480_input.iloc[14:16].depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta$D$_{C30}$ tie')
# benthic d18O tie points
plt.errorbar(dsdp480_input.iloc[16].age-dsdp480_input.iloc[16].dR, dsdp480_input.iloc[16].depth, 
             xerr=xerr[16]*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp480_input.iloc[16].age, dsdp480_input.iloc[16].depth,
            marker='^', color='lightblue', **scat_kw, label='_Hidden')
# DSDP-479 tie-points
plt.scatter(d480_479_tiepoint[1][0], d480_479_tiepoint[1][1],
            marker='x', color='r', **scat_kw, label='DSDP-479/480 $\delta$D$_{C30}$ tie')
plt.scatter(d480_479_tiepoint[0][0], d480_479_tiepoint[0][1],
                marker='x', color='coral', **scat_kw, label='DSDP-479/480 pollen tie')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.text(135000, 100, 'DSDP 480', **title_text_kw)
ax.set(xlim=[1,137000], ylim=[5000,0])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.set_ylabel("Core Depth (cm)")
ax.set_xticks([25000,50000,75000,100000,125000])
ax.set_xticklabels([25,50,75,100,125])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False, **tkw)
ax.legend(**legend_kw)


# DSDP-479
ax = plt.subplot2grid((3, 2), (0, 1), rowspan=2, colspan=1) #plt.subplot(122)
ax.text(97500, 3290, 'B', **title_text_kw)
# 2-sigma bounds and shading
ax.plot(dsdp479_agedepth[iupperupper, :], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(dsdp479_agedepth[ilowerlower, :], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.fill_betweenx(dsdp479_agedepth.depth,
                 dsdp479_agedepth[ilowerlower, :],
                 dsdp479_agedepth[iupperupper, :],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence interval')
# 1-sigma shading
ax.fill_betweenx(dsdp479_agedepth.depth,
                 dsdp479_agedepth[iouter, :],
                 dsdp479_agedepth[iinner, :],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence interval')
# median line
ax.plot(median479, dsdp479_agedepth.depth, '-', c='k', lw=1, label='median')
# d479-LR04 tie points
plt.errorbar(dsdp479_input.age.values-dsdp479_input.dR.values, dsdp479_input.depth.values, 
             xerr=xerr479*2, yerr=0, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp479_input.age.values, dsdp479_input.depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta$D$_{C30}$ tie')
# DSDP-480 tie-points
plt.scatter(d479_480_tiepoint[0][0], d479_480_tiepoint[0][1],
                marker='x', color='coral', **scat_kw, label='DSDP-480 pollen tie') #
plt.scatter(d479_480_tiepoint[1][0], d479_480_tiepoint[1][1],
            marker='x', color='r', **scat_kw, label='DSDP-480 $\delta$D$_{C30}$ tie') #
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.text(159000, 3490, 'DSDP 479', **title_text_kw)
ax.set(xlim=[100000,160000], ylim=[5750,3450])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.yaxis.set_label_position('right')
ax.set_ylabel(" ", rotation=270, labelpad=15)
ax.set_xticks([100000,120000,140000,160000])
ax.set_xticklabels([100,120,140,160])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False,
               right=True, labelright=True, **tkw)



### Timeseries
line_kw={'ls':'-', 'lw':1.5} #, 'marker':'s', 'markersize':4, 'mec':'k', 'mew':0.25, 'zorder':100} 
patch_kw = {'ec':'k', 'lw':1, 'linestyle':':', 'fc':'grey', 'alpha':0.25}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'zorder': 100, 'clip_on': False}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
arrow_kw={'arrowstyle':'->', 'color':'k', 'linewidth':2, 'clip_on':False}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':8, 'weight':'bold', 'color':'k', 'ha':'left', 'va':'center'} #'backgroundcolor':'white', 
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
legend_kw = {'loc':'upper right', 'bbox_to_anchor':(1, 1), 'fontsize':7, 'labelcolor':'k', 'frameon':True, 'facecolor':'w', 'framealpha':1}


ax1 = plt.subplot2grid((3, 2), (2, 0), rowspan=1, colspan=2)
ax1.text(-7.5, 2.1, 'C', **title_text_kw)
pp=plt.Rectangle((0, hol_d18o_min), 11.7, hol_d18o_max-hol_d18o_min, zorder=100, label='_Hidden', clip_on=True, **patch_kw) 
ax1.add_patch(pp)
ax1.annotate('Expected Holocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$',
             xy=(11.7,2.3), xytext=(25,2.3),
             arrowprops=arrow_kw, **label_text_kw)
#ax1.annotate('Expected\nHolocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$',
#             xy=(0,hol_d18o_min), xytext=(0,hol_d18o_max-.05),
#             arrowprops=arrow_kw, **label_text_kw, zorder=100)
ax1.plot(sh_d18o['age']/1000, sh_d18o['dat'], c='lightblue', **line_kw, label='SH82')
ax1.plot(kj_d18o['age']/1000, kj_d18o['dat'], c='lightsteelblue', **line_kw, label='KJ90')
ax1.plot(lr04['age'], lr04['dat'], c='k', lw=2, label='LR04') # benthic stack
ax1.scatter(tie_points['d18o_this_study'].d18o.age, tie_points['d18o_this_study'].d18o,
            marker='o', color='peru', **scat_kw, label='tie, this study')
ax1.errorbar(tie_points['d18o_this_study'].d18o.age, tie_points['d18o_this_study'].d18o, 
             xerr=0, yerr=0.05, fmt='^', color='none', ecolor='k', capsize=2,label='_Hidden')
ax1.scatter(d480_ages, tie_points['d18o'].sh_d18o,
            marker='x', color='red', **scat_kw, label='tie, SH82')
ax1.scatter(tie_points['d18o'].lr04_d18o.age, tie_points['d18o'].lr04_d18o,
            marker='x', color='red', **scat_kw, label='_Hidden')
ax1.set(xlim=[xmin,xmax], ylim=[5.2,2.1])
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.set_ylabel(u'$\delta^{18}O_{benthic}$ [‰]', labelpad=5, **laxis_text_kw)
ax1.set_xlabel('AGE (ka)')

ax2=ax1.twinx() 
l6, =ax2.plot(by_aj['age']/1000, by_aj['dat'], c='grey', ls='-.', lw=1.25, label='Byrne90') # art+jun pollen
ax2.set(xlim=[xmin,xmax], ylim=[33.5,-2], yticks=[0,10,20,30])
ax2.minorticks_on()
ax2.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax2.set_ylabel(u'%Artemisia+Juniper', labelpad=15, **raxis_text_kw)
ax2.patch.set_visible(False)
ax2.spines['right'].set_color('grey')

ax1.legend(**legend_kw)

plt.savefig(f'{opath}/dsdp_480-479.agemodel.pdf', bbox_inches='tight')